In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/compas_clean.csv")
df = df[df["race"].isin(["African-American", "Caucasian"])].reset_index(drop=True)

X = pd.get_dummies(df[["age", "priors_count", "c_charge_degree", "sex"]],
                   drop_first=True, dtype=int)
y = df["two_year_recid"]

train_idx, test_idx = train_test_split(
    df.index, test_size=0.3, random_state=42, stratify=y
)
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
race_train, race_test = df.loc[train_idx, "race"], df.loc[test_idx, "race"]
X.head()

,age,priors_count,c_charge_degree_M,sex_Male
0,34,0,0,1
1,24,4,0,1
2,41,14,0,1
3,39,0,1,0
4,27,0,0,1


In [2]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05).fit(X_train, y_train)

for name, model in [("Logistic Regression", logreg), ("XGBoost", xgb)]:
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(name)
    print("  accuracy:", round(accuracy_score(y_test, pred), 3))
    print("  AUC:", round(roc_auc_score(y_test, proba), 3))
    print(confusion_matrix(y_test, pred))
    print()

Logistic Regression
  accuracy: 0.67
  AUC: 0.721
[[627 212]
 [310 435]]

XGBoost
  accuracy: 0.672
  AUC: 0.725
[[616 223]
 [297 448]]



In [3]:
compas_pred = (df.loc[test_idx, "decile_score"] >= 5).astype(int)
print("COMPAS")
print("  accuracy:", round(accuracy_score(y_test, compas_pred), 3))
print(confusion_matrix(y_test, compas_pred))

COMPAS
  accuracy: 0.668
[[580 259]
 [267 478]]


## Model Performance Comparison & Takeaways

| Model | Accuracy | AUC | False Positives | False Negatives | FPR (FP / 839) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 0.670 | 0.721 | 212 | 310 | 25.3% |
| **XGBoost** | 0.672 | 0.725 | 223 | 297 | 26.6% |
| **COMPAS (score of 5 or above)** | 0.668 | N/A | 259 | 267 | 30.9% |

*The test set has 1,584 defendants. 839 of them were not rearrested, and that is the denominator for the false positive rate (FPR).*

---

### Key Takeaways

1. **Similar accuracy across all three models.**
   Overall accuracy is about **67%** for every model. XGBoost's edge over logistic regression is 0.2 percentage points, roughly 3 defendants out of 1,584. That difference is too small to treat as real, so the added complexity gives no meaningful gain.

2. **A simple model matches COMPAS.**
   Logistic regression using just four inputs (`age`, `priors_count`, `c_charge_degree`, and `sex`) performs on par with the proprietary COMPAS score.

3. **The models make different kinds of errors.**
   * **COMPAS** flags more people, giving the most false positives (259, FPR about 31%) but the fewest false negatives (267).
   * **Logistic regression** has the fewest false positives (212, FPR about 25%) but misses more people who were rearrested (310 false negatives).
   * **XGBoost** sits in the middle (223 false positives, 297 false